# Cell Database Agent — Notebook Test

Run each cell in order. The notebook is divided into 6 stages:
1. Verify h5ad loads correctly
2. Test the NaN-cleaning fix
3. Mini build (100-cell subset) — validates the full pipeline quickly
4. Verify the database tables and counts
5. Test SQL filter queries
6. Test the agent end-to-end query

In [1]:
import sys
import os

# ── Edit these paths to match your environment ──────────────────────────────
SCRIPT_DIR  = '/home/zhuoxuan/C2S/cell_data_base'        # where the 4 .py files live
H5AD_PATH   = '/home/zhuoxuan/C2S/Guts/guts_data/guts_preprocessed_data.h5ad'
EMBED_MODEL = '/nfs/turbo/umms-drjieliu/proj/c2s_model/C2S-Pythia-410m-cell-type-prediction'
NL_MODEL    = '/home/zhuoxuan/c2s_model/C2S-Scale-Pythia-1b-pt'
TEST_DB     = '/tmp/test_cell_db/test.db'                 # throwaway DB for notebook tests
TEST_ARROW  = '/tmp/test_cell_db/arrow_ds'

sys.path.insert(0, SCRIPT_DIR)
os.makedirs(os.path.dirname(TEST_DB), exist_ok=True)
print('Script dir on path:', SCRIPT_DIR)
print('All paths set.')

Script dir on path: /home/zhuoxuan/C2S/cell_data_base
All paths set.


## Stage 1 — Verify the h5ad loads correctly

In [2]:
import anndata
import pandas as pd

adata = anndata.read_h5ad(H5AD_PATH)
print(f'Shape: {adata.n_obs} cells × {adata.n_vars} genes')
print(f'\nobs columns ({len(adata.obs.columns)}):')
print(list(adata.obs.columns))
print(f'\nFirst 5 var_names: {list(adata.var_names[:5])}')
print(f'Last  5 var_names: {list(adata.var_names[-5:])}')

Shape: 1168 cells × 20012 genes

obs columns (29):
['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'RNA_snn_res.0.1', 'seurat_clusters', 'cell_type', 'batch_condition', 'RNA_snn_res.1', 'RNA_snn_res.0.05', 'RNA_snn_res.0.2', 'RNA_snn_res.0.35', 'unintegrated_clusters', 'Age', 'Tissue', 'region', 'SampleType', 'tissue', 'RNA_snn_res.3.5', 'RNA_snn_res.2', 'gene', 'organism', 'assay', 'sex', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']

First 5 var_names: ['AL627309.1', 'AL627309.5', 'LINC01409', 'LINC01128', 'LINC00115']
Last  5 var_names: ['AL592183.1', 'AC240274.1', 'AC004556.3', 'AC007325.4', 'AC007325.2']


In [3]:
print('Column dtype and NaN count:')
print(f'{"Column":<30} {"dtype":<12} {"NaN count"}')
print('-' * 55)
for col in adata.obs.columns:
    nan_count = adata.obs[col].isna().sum()
    print(f'{col:<30} {str(adata.obs[col].dtype):<12} {nan_count}')

Column dtype and NaN count:
Column                         dtype        NaN count
-------------------------------------------------------
orig.ident                     category     0
nCount_RNA                     int64        0
nFeature_RNA                   int64        0
percent.mt                     float64      0
RNA_snn_res.0.1                float64      1168
seurat_clusters                int64        0
cell_type                      category     0
batch_condition                category     0
RNA_snn_res.1                  int64        0
RNA_snn_res.0.05               float64      1168
RNA_snn_res.0.2                float64      1135
RNA_snn_res.0.35               float64      1135
unintegrated_clusters          int64        0
Age                            category     0
Tissue                         category     392
region                         category     392
SampleType                     category     0
tissue                         category     392
RNA_snn_res.3.5 

## Stage 2 — Test the NaN-cleaning fix

This replicates the exact cleaning step that was failing in `build_database.py`.

In [4]:
from build_database import OBS_ALIASES, C2S_LABEL_KEYS

# Resolve column aliases (same logic as build_database.py)
def _resolve(obs_df, aliases):
    for a in aliases:
        if a in obs_df.columns:
            return a
    return None

col_map = {key: _resolve(adata.obs, aliases) for key, aliases in OBS_ALIASES.items()}

print('Column mapping:')
for k, v in col_map.items():
    print(f'  {k:<25} → {v}')

c2s_label_cols = [col_map[k] for k in C2S_LABEL_KEYS if col_map.get(k)]
print(f'\nC2S label columns: {c2s_label_cols}')

Column mapping:
  cell_type                 → cell_type
  tissue                    → Tissue
  batch_condition           → batch_condition
  sample_type               → SampleType
  region                    → region
  orig_ident                → orig.ident
  age                       → Age
  organism                  → organism
  assay                     → assay
  sex                       → sex
  n_count_rna               → nCount_RNA
  n_feature_rna             → nFeature_RNA
  percent_mt                → percent.mt
  gene                      → gene
  n_genes                   → n_genes
  n_genes_by_counts         → n_genes_by_counts
  total_counts              → total_counts
  total_counts_mt           → total_counts_mt
  pct_counts_mt             → pct_counts_mt
  seurat_clusters           → seurat_clusters
  unintegrated_clusters     → unintegrated_clusters

C2S label columns: ['cell_type', 'Tissue', 'batch_condition', 'organism', 'sex', 'region']


In [5]:
obs_c2s = adata.obs.copy()
NAN_STRINGS = {'nan', 'none', 'nat', '<na>', ''}

for key in C2S_LABEL_KEYS:
    actual_col = col_map.get(key)
    if actual_col and actual_col in obs_c2s.columns:
        obs_c2s[actual_col] = (
            obs_c2s[actual_col]
            .astype(str)
            .apply(lambda v: 'NA' if v.strip().lower() in NAN_STRINGS else v)
        )

print('After cleaning — remaining float NaNs in label columns:')
all_clean = True
for col in c2s_label_cols:
    nan_count = obs_c2s[col].isna().sum()
    float_count = sum(1 for v in obs_c2s[col] if isinstance(v, float))
    status = 'OK' if nan_count == 0 and float_count == 0 else 'PROBLEM'
    print(f'  [{status}] {col:<25} NaN={nan_count}  floats={float_count}')
    if status == 'PROBLEM':
        all_clean = False

print()
if all_clean:
    print('All label columns are clean strings. adata_to_arrow should succeed.')
else:
    print('WARNING: Some columns still have issues. Check above.')

After cleaning — remaining float NaNs in label columns:
  [OK] cell_type                 NaN=0  floats=0
  [OK] Tissue                    NaN=0  floats=0
  [OK] batch_condition           NaN=0  floats=0
  [OK] organism                  NaN=0  floats=0
  [OK] sex                       NaN=0  floats=0
  [OK] region                    NaN=0  floats=0

All label columns are clean strings. adata_to_arrow should succeed.


## Stage 3 — Mini build (100-cell subset)

Runs the full `build_database.py` pipeline on 100 randomly sampled cells.
This is fast (~2 min on CPU) and validates every step end-to-end before submitting the full sbatch job.

In [6]:
import random
import numpy as np
import cell2sentence as cs
from cell2sentence.tasks import embed_cells
import database as db
from database import SEURAT_RES_COLS
from build_database import build_row, C2S_LABEL_KEYS, OBS_ALIASES

random.seed(1234)
np.random.seed(1234)

# Sample 100 cells
N = 100
idx = random.sample(range(adata.n_obs), N)
adata_mini = adata[idx].copy()
print(f'Mini dataset: {adata_mini.n_obs} cells × {adata_mini.n_vars} genes')

Mini dataset: 100 cells × 20012 genes


In [7]:
# Clean label columns
obs_mini = adata_mini.obs.copy()
NAN_STRINGS = {'nan', 'none', 'nat', '<na>', ''}
for key in C2S_LABEL_KEYS:
    actual_col = col_map.get(key)
    if actual_col and actual_col in obs_mini.columns:
        obs_mini[actual_col] = (
            obs_mini[actual_col]
            .astype(str)
            .apply(lambda v: 'NA' if v.strip().lower() in NAN_STRINGS else v)
        )

adata_mini_c2s = adata_mini.copy()
adata_mini_c2s.obs = obs_mini

print('Converting to Arrow format...')
arrow_ds, vocabulary = cs.CSData.adata_to_arrow(
    adata=adata_mini_c2s,
    random_state=1234,
    sentence_delimiter=' ',
    label_col_names=c2s_label_cols,
)
print(f'Arrow dataset: {arrow_ds.num_rows} rows')
print(f'Sample row keys: {list(arrow_ds[0].keys())}')
print(f'\nFirst cell sentence (first 20 genes):')
print(' '.join(arrow_ds[0]['cell_sentence'].split()[:20]))

WARN: more variables (20012) than observations (100)... did you mean to transpose the object (e.g. adata.T)?
WARN: more variables (20012) than observations (100), did you mean to transpose the object (e.g. adata.T)?


Converting to Arrow format...


100%|██████████| 100/100 [00:00<00:00, 2177.38it/s]

Arrow dataset: 100 rows
Sample row keys: ['cell_name', 'cell_sentence', 'cell_type', 'Tissue', 'batch_condition', 'organism', 'sex', 'region']

First cell sentence (first 20 genes):
MALAT1 MT-CO1 EEF1A1 GHRL MT-CO3 MT-CO2 KCNB2 RPL13 RPS3 RPS8 RPS12 TXN TPT1 MT-ATP6 RPLP1 RPS14 RPL10 RPL34 RPL41 RPS2


In [8]:
import tempfile, shutil

csdata_dir = '/tmp/test_cell_db/csdata'
os.makedirs(csdata_dir, exist_ok=True)

csdata = cs.CSData.csdata_from_arrow(
    arrow_dataset=arrow_ds,
    vocabulary=vocabulary,
    save_dir=csdata_dir,
    save_name='mini_csdata',
    dataset_backend='arrow',
)

sentences = csdata.get_sentence_strings()
print(f'Got {len(sentences)} sentences')
print(f'\nExample sentence (first 15 genes):')
print(' '.join(sentences[0].split()[:15]))

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Got 100 sentences

Example sentence (first 15 genes):
MALAT1 MT-CO1 EEF1A1 GHRL MT-CO3 MT-CO2 KCNB2 RPL13 RPS3 RPS8 RPS12 TXN TPT1 MT-ATP6 RPLP1


In [9]:
print('Loading embedding model (this may take a minute)...')
csmodel = cs.CSModel(
    model_name_or_path=EMBED_MODEL,
    save_dir='/tmp/test_cell_db/csmodel_cache',
    save_name='embed_model',
)

print('Embedding 100 cells...')
embeddings = embed_cells(csdata=csdata, csmodel=csmodel, n_genes=200).astype('float32')
print(f'Embeddings shape: {embeddings.shape}')
print(f'Min: {embeddings.min():.4f}  Max: {embeddings.max():.4f}  Mean: {embeddings.mean():.4f}')

Loading embedding model (this may take a minute)...
Using device: cuda
Embedding 100 cells...
Reloading model from path on disk: /tmp/test_cell_db/csmodel_cache/embed_model
Embedding 100 cells using CSModel...


100%|██████████| 100/100 [00:02<00:00, 43.93it/s]

Embeddings shape: (100, 1024)
Min: -11.0189  Max: 21.2212  Mean: -0.0061


In [10]:
# Initialise DB and insert genes
if os.path.exists(TEST_DB):
    os.remove(TEST_DB)

db.init_db(TEST_DB)

gene_records = [{
    'ensembl_id': str(adata_mini.var.loc[g, 'ensembl_id']) if 'ensembl_id' in adata_mini.var.columns else '',
    'gene_name': str(g),
    'is_mitochondrial': int(str(g).startswith('MT-'))
} for g in adata_mini.var_names]
db.insert_genes(TEST_DB, gene_records)

# Build row dicts
barcodes = adata_mini.obs_names.tolist()
rows = [build_row(bc, adata_mini.obs.iloc[i], col_map) for i, bc in enumerate(barcodes)]

# Bulk insert
db.insert_cells_batch(
    db_path=TEST_DB,
    rows=rows,
    embeddings=embeddings,
    sentences=sentences,
    model_name='C2S-Pythia-410m-mini-test',
    n_genes=200,
)
print('Mini DB built successfully.')

INFO | Database initialised: /tmp/test_cell_db/test.db
INFO | Inserted 20012 gene records.
INFO | Inserting 100 cells …
INFO |   … 100/100 committed
INFO | All cells inserted.


Mini DB built successfully.


## Stage 4 — Verify database tables and counts

In [11]:
import sqlite3

conn = sqlite3.connect(TEST_DB)
conn.row_factory = sqlite3.Row

tables = [
    'cells', 'cell_embeddings', 'cell_sentences', 'cell_cluster_assignments',
    'genes', 'cell_types', 'tissues', 'batches', 'organisms', 'assays', 'sexes',
    'regions', 'sample_types', 'orig_idents', 'ages'
]

print(f'{"Table":<30} {"Row count"}')
print('-' * 42)
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:<30} {n}')
conn.close()

Table                          Row count
------------------------------------------
cells                          100
cell_embeddings                100
cell_sentences                 100
cell_cluster_assignments       308
genes                          20012
cell_types                     1
tissues                        4
batches                        3
organisms                      1
assays                         1
sexes                          0
regions                        2
sample_types                   1
orig_idents                    19
ages                           8


In [12]:
conn = sqlite3.connect(TEST_DB)
conn.row_factory = sqlite3.Row

dim_tables = ['cell_types', 'tissues', 'batches', 'organisms', 'sexes', 'assays', 'regions', 'ages']
for t in dim_tables:
    rows = conn.execute(f'SELECT name FROM {t} ORDER BY name').fetchall()
    vals = [r['name'] for r in rows]
    print(f'{t:<15}: {vals}')

conn.close()

cell_types     : ['EECs']
tissues        : ['colon', 'duodenum', 'ileum', 'jejunum']
batches        : ['PMID31753849', 'PMID34497389', 'PMID35176508']
organisms      : ['Homo sapiens']
sexes          : []
assays         : ['scRNA-seq']
regions        : ['large intestine', 'small intestine']
ages           : ['25 to 30 years', '29 years', '45 to 50 years', '45 years', '53 years', '55 years', '61 years', '67 years']


In [13]:
conn = sqlite3.connect(TEST_DB)
conn.row_factory = sqlite3.Row

print('Cluster assignments (first 10 rows):')
rows = conn.execute(
    'SELECT cell_id, resolution, cluster_id FROM cell_cluster_assignments ORDER BY cell_id, resolution LIMIT 10'
).fetchall()
print(f'{"cell_id":<10} {"resolution":<12} {"cluster_id"}')
print('-' * 35)
for r in rows:
    print(f"{r['cell_id']:<10} {r['resolution']:<12} {r['cluster_id']}")

resolutions = [r[0] for r in conn.execute(
    'SELECT DISTINCT resolution FROM cell_cluster_assignments ORDER BY resolution'
).fetchall()]
print(f'\nDistinct resolutions stored: {resolutions}')
conn.close()

Cluster assignments (first 10 rows):
cell_id    resolution   cluster_id
-----------------------------------
1          1.0          37
1          2.0          47
1          3.5          57
2          1.0          37
2          2.0          47
2          3.5          57
3          1.0          37
3          2.0          47
3          3.5          57
4          1.0          37

Distinct resolutions stored: [0.2, 0.35, 1.0, 2.0, 3.5]


In [14]:
conn = sqlite3.connect(TEST_DB)
conn.row_factory = sqlite3.Row

row = conn.execute('SELECT cell_id, embedding, embedding_dim, model_name FROM cell_embeddings LIMIT 1').fetchone()
emb = np.frombuffer(row['embedding'], dtype=np.float32)
print(f'cell_id       : {row["cell_id"]}')
print(f'embedding_dim : {row["embedding_dim"]}')
print(f'model_name    : {row["model_name"]}')
print(f'embedding[:8] : {emb[:8]}')
print(f'L2 norm       : {np.linalg.norm(emb):.4f}')

conn.close()

cell_id       : 1
embedding_dim : 1024
model_name    : C2S-Pythia-410m-mini-test
embedding[:8] : [ 2.1893117   0.02274383  1.832599    0.16651906 -0.74621415  2.527785
 -1.6220614   0.3123318 ]
L2 norm       : 56.8337


## Stage 5 — Test SQL filter queries

In [15]:
catalogue = db.get_catalogue(TEST_DB)
print('Live catalogue from DB:')
for dim, vals in catalogue.items():
    print(f'  {dim:<22}: {vals}')

Live catalogue from DB:
  cell_types            : ['EECs']
  tissues               : ['colon', 'duodenum', 'ileum', 'jejunum']
  batches               : ['PMID31753849', 'PMID34497389', 'PMID35176508']
  sample_types          : ['adult']
  regions               : ['large intestine', 'small intestine']
  orig_idents           : ['Human_colon_16S8000513', 'Human_colon_16S8001903', 'Human_colon_16S8123911', 'SRR16569986', 'SRR16569987', 'SRR16569988', 'SRR16569989', 'SRR16569994', 'SRR16569995', 'SRR16569996', 'SRR16569997', 'SRR16570002', 'SRR16570003', 'SRR16570004', 'SRR16570005', 'SRR8513794', 'SRR8513795', 'SRR8513796', 'SRR8513797']
  ages                  : ['25 to 30 years', '29 years', '45 to 50 years', '45 years', '53 years', '55 years', '61 years', '67 years']
  organisms             : ['Homo sapiens']
  assays                : ['scRNA-seq']
  sexes                 : []
  cluster_resolutions   : [0.2, 0.35, 1.0, 2.0, 3.5]


In [16]:
# Fetch all cells — no filters
all_cells = db.fetch_cells_by_filters(TEST_DB)
print(f'All cells (no filter): {len(all_cells)}')
print(f'Keys: {list(all_cells[0].keys())}')
print(f'\nFirst row:')
for k, v in all_cells[0].items():
    print(f'  {k:<25}: {v}')

All cells (no filter): 100
Keys: ['cell_id', 'barcode', 'cell_type', 'tissue', 'batch_condition', 'sample_type', 'region', 'orig_ident', 'age', 'organism', 'assay', 'sex', 'n_count_rna', 'n_feature_rna', 'percent_mt', 'gene', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'seurat_clusters', 'unintegrated_clusters']

First row:
  cell_id                  : 1
  barcode                  : PMID35176508_SRR16570003_CCACTTGCATCCGCGA-1
  cell_type                : EECs
  tissue                   : jejunum
  batch_condition          : PMID35176508
  sample_type              : adult
  region                   : small intestine
  orig_ident               : SRR16570003
  age                      : 53 years
  organism                 : Homo sapiens
  assay                    : scRNA-seq
  sex                      : None
  n_count_rna              : 15526.0
  n_feature_rna            : 4942
  percent_mt               : 3.2783717634935
  gene                     

In [17]:
# Get unique cell types in mini DB to pick one that exists
catalogue = db.get_catalogue(TEST_DB)
cell_types = catalogue['cell_types']
print(f'Cell types in mini DB: {cell_types}')

if cell_types:
    test_ct = cell_types[0]
    results = db.fetch_cells_by_filters(TEST_DB, cell_type=test_ct)
    print(f'\nFilter: cell_type="{test_ct}" → {len(results)} cells')
    if results:
        print(f'  Sample: barcode={results[0]["barcode"]}  tissue={results[0]["tissue"]}')

Cell types in mini DB: ['EECs']

Filter: cell_type="EECs" → 100 cells
  Sample: barcode=PMID35176508_SRR16570003_CCACTTGCATCCGCGA-1  tissue=jejunum


In [18]:
resolutions = catalogue['cluster_resolutions']
print(f'Available resolutions: {resolutions}')

if resolutions:
    res = resolutions[0]
    # Get all cluster IDs at this resolution
    conn = sqlite3.connect(TEST_DB)
    cluster_ids = [r[0] for r in conn.execute(
        'SELECT DISTINCT cluster_id FROM cell_cluster_assignments WHERE resolution=? ORDER BY cluster_id',
        (res,)
    ).fetchall()]
    conn.close()
    print(f'Cluster IDs at resolution {res}: {cluster_ids}')

    if cluster_ids:
        cid = cluster_ids[0]
        results = db.fetch_cells_by_filters(
            TEST_DB, cluster_resolution=res, cluster_id=cid
        )
        print(f'\nFilter: resolution={res}, cluster_id={cid} → {len(results)} cells')

Available resolutions: [0.2, 0.35, 1.0, 2.0, 3.5]
Cluster IDs at resolution 0.2: [2, 3]

Filter: resolution=0.2, cluster_id=2 → 1 cells


In [19]:
# Get the actual percent_mt range in the mini DB
conn = sqlite3.connect(TEST_DB)
stats = conn.execute('SELECT MIN(percent_mt), MAX(percent_mt), AVG(percent_mt) FROM cells').fetchone()
conn.close()
print(f'percent_mt — min: {stats[0]:.2f}  max: {stats[1]:.2f}  mean: {stats[2]:.2f}')

threshold = stats[2]   # use mean as cutoff
results = db.fetch_cells_by_filters(TEST_DB, percent_mt_max=threshold)
print(f'\nFilter: percent_mt <= {threshold:.2f} → {len(results)} cells')

percent_mt — min: 1.78  max: 39.65  mean: 11.18

Filter: percent_mt <= 11.18 → 63 cells


## Stage 6 — Test the agent end-to-end

In [20]:
# Save the mini arrow_ds so CellDatabaseAgent can load it
arrow_ds.save_to_disk(TEST_ARROW)
print(f'Arrow dataset saved to {TEST_ARROW}')

print('\nLoading CellDatabaseAgent (loads NL model — may take ~1 min)...')
from cell_db_agent import CellDatabaseAgent, CellQuery

agent = CellDatabaseAgent(
    db_path=TEST_DB,
    arrow_dir=TEST_ARROW,
    nl_model_path=NL_MODEL,
    top_k_genes=200,
    n_neighbours=5,
    n_cells_per_prompt=5,
)
print('Agent ready.')

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

INFO | Device: cuda
INFO | Loading Arrow dataset from /tmp/test_cell_db/arrow_ds
INFO | Loading NL model from /home/zhuoxuan/c2s_model/C2S-Scale-Pythia-1b-pt


Arrow dataset saved to /tmp/test_cell_db/arrow_ds

Loading CellDatabaseAgent (loads NL model — may take ~1 min)...
Using device: cuda


INFO | NL model ready.


Agent ready.


In [21]:
# Pick a cell type that exists in the mini DB
ct = db.get_catalogue(TEST_DB)['cell_types'][0]
print(f'Querying by cell type: "{ct}"')

query = CellQuery(cell_type=ct)
print(f'Query mode: {query.query_mode}')

result = agent.query(query)
print('\n=== C2S output ===')
print(result)

INFO | Query mode: by_cell_type | {'cell_type': 'EECs'}
INFO | Top-5 cell IDs: [20, 34, 40, 51, 92]…


Querying by cell type: "EECs"
Query mode: by_cell_type

=== C2S output ===
This study used single-cell technologies to map human intestinal development, revealing the differentiation hierarchies of progenitor populations and the origins of immune cells. The study also identified the formation of crypt-villus axis, neural, vascular, and immune populations, and cell development beyond the crypt-villus axis. The findings offer insights into human intestinal development and associated disorders..


In [22]:
from chat import CatalogueMatcher

catalogue = agent.get_catalogue()
matcher = CatalogueMatcher(catalogue)

test_messages = [
    f'tell me about {ct}',
    f'cells from resolution 0.1 cluster 0',
    'cells with percent mt < 10',
    'seurat cluster 1',
]

print('CatalogueMatcher parsing test:')
print('=' * 50)
for msg in test_messages:
    q = matcher.parse_query(msg)
    print(f'\nInput : "{msg}"')
    print(f'Mode  : {q.query_mode}')
    print(f'Filters: {q.active_filters()}')

CatalogueMatcher parsing test:

Input : "tell me about EECs"
Mode  : by_cell_type
Filters: {'cell_type': 'EECs'}

Input : "cells from resolution 0.1 cluster 0"
Mode  : by_metadata
Filters: {'batch_condition': 'PMID35176508', 'orig_ident': 'Human_colon_16S8000513', 'age': '25 to 30 years', 'cluster_resolution': 0.1, 'cluster_id': 0}

Input : "cells with percent mt < 10"
Mode  : by_metadata
Filters: {'percent_mt_max': 10.0}

Input : "seurat cluster 1"
Mode  : by_metadata
Filters: {'batch_condition': 'PMID31753849', 'orig_ident': 'Human_colon_16S8000513', 'age': '61 years', 'seurat_cluster': 1}


In [23]:
from chat import C2SChatbot

chatbot = C2SChatbot(agent=agent, max_new_tokens=256)

msg = f'what can you tell me about {ct}'
print(f'You: {msg}')
response = chatbot.chat(msg)
print(f'\nAgent: {response}')

INFO | Query mode: by_cell_type | {'cell_type': 'EECs'}
INFO | Top-5 cell IDs: [20, 34, 40, 51, 92]…


You: what can you tell me about EECs
  [parser] mode=by_cell_type | {'cell_type': 'EECs'}

Agent: right colon.
.
Cell-type: enteroendocrine cell.
Cell 1: CDX1 MLOX1 CDCA4 PYY ACTG1 RBP4 PP7080 LGR5 ACTN4 CD177 CPA6 TFF2 PPP4R1L TCF12 GTF2A1L TSPAN13 GCG UG0808D-UPP2 CELA2A RNF10 RAB11FIP3 TPPP3 FNDC3B CFL1 NUMA1 FMR1 TENT5A TTC26 BTF3L4 TSPAN3 NEDD4L FARP1 TENT4A GATA6-AS1 NEDD9 TENT4B TMEM212 UBA5 UPP1 UBE2D3 NUFSP2 C3orf50 TMEM100 MAML3 LARP1B RNF11 ZNFX1 CFLAR RBM3 RBMX2 TNRC18 ZNFX1 FMR1-AS1 KATNBL1 ZNF212 SLC25A11 RP11-295D4-1 UBE3A RAB11FIP2 C
